# Influence of homophily and heterophily on oversmoothing of GAT and TransformerConv

## 1 High homophily graphs

### 1.1 pubmed dataset

In [2]:
import torch
import torch_geometric
from typing import Any
import itertools

c:\Users\Mislav.FERIT-PC\Desktop\oversmoothing-GAT-TransformerConv\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] device in use {DEVICE}")

[INFO] device in use cuda


In [4]:
path_pubmed = "./data/PubMed"

In [5]:
pubmed_dataset = torch_geometric.datasets.Planetoid(
    root=path_pubmed,
    name="PubMed",
    transform=torch_geometric.transforms.NormalizeFeatures()
)

In [6]:
print("Dataset information")
print("===================")
print(f"Number of graphs in dataset: {len(pubmed_dataset)}")
print(f"Number of features: {pubmed_dataset.num_features}")
print(f"Number of classes: {pubmed_dataset.num_classes}")

Dataset information
Number of graphs in dataset: 1
Number of features: 500
Number of classes: 3


In [7]:
pubmed_data = pubmed_dataset[0]
print(pubmed_data)

Data(x=[19717, 500], edge_index=[2, 88648], y=[19717], train_mask=[19717], val_mask=[19717], test_mask=[19717])


In [8]:
print("Graph information")
print("==================")
print(f"Number of nodes: {pubmed_data.num_nodes}")
print(f"Number of edges: {pubmed_data.num_edges}")
print(f"Has isolated nodes: {pubmed_data.has_isolated_nodes()}")
print(f"Has self loops: {pubmed_data.has_self_loops()}")

Graph information
Number of nodes: 19717
Number of edges: 88648
Has isolated nodes: False
Has self loops: False


## 2 GAT network definition

In [9]:
class GATNetwork(torch.nn.Module):
    def __init__(
        self,
        number_of_layers: int,
        input_size: int,
        hidden_size: int,
        output_size: int,
        num_heads: int,
        dropout_rate: float = 0.5
    ):
        super().__init__()

        self.layers = torch.nn.ModuleList()

        input_channels = input_size
        output_channels = hidden_size
        for i in range(number_of_layers - 1):
            self.layers.append(
                torch_geometric.nn.LayerNorm(in_channels=input_channels)
            )
            self.layers.append(
                torch_geometric.nn.GATv2Conv(in_channels=input_channels, out_channels=output_channels, heads=num_heads)
            )
            self.layers.append(
                torch.nn.Dropout(p=dropout_rate)
            )
            self.layers.append(
                torch.nn.GELU()
            )
            input_channels = hidden_size * num_heads

        output_channels = output_size
        self.layers.append(
            torch_geometric.nn.LayerNorm(in_channels=input_channels)
        )
        self.layers.append(torch_geometric.nn.GATv2Conv(in_channels=input_channels, out_channels=output_channels, heads=num_heads, concat=False))
        
        self.layers = torch.nn.ModuleList(self.layers)
        
    def forward(self, x: torch.Tensor, edge_index: torch.Tensor):
        for layer in self.layers:
            if isinstance(layer, torch_geometric.nn.GATv2Conv):
                x = layer(x, edge_index)
            else:
                x = layer(x)
        return x

## 3 TransformerConv network definition

In [10]:
class TransformerConvNetwork(torch.nn.Module):
    def __init__(
        self,
        number_of_layers: int,
        input_size: int,
        hidden_size: int,
        output_size: int,
        num_heads: int,
        dropout_rate: float = 0.5
    ):
        super().__init__()

        self.layers = torch.nn.ModuleList()

        input_channels = input_size
        output_channels = hidden_size
        for i in range(number_of_layers - 1):
            self.layers.append(
                torch_geometric.nn.LayerNorm(in_channels=input_channels)
            )
            self.layers.append(
                torch_geometric.nn.TransformerConv(in_channels=input_channels, out_channels=output_channels, heads=num_heads, beta=True)
            )
            self.layers.append(
                torch.nn.Dropout(p=dropout_rate),
            )
            self.layers.append(
                torch.nn.GELU()
            )
            input_channels = hidden_size * num_heads
            
        output_channels = output_size
        self.layers.append(
            torch_geometric.nn.LayerNorm(in_channels=input_channels)
        )
        self.layers.append(torch_geometric.nn.TransformerConv(
            in_channels = input_channels,
            out_channels = output_channels,
            heads = num_heads,
            concat = False,
            beta = True
        ))
    
    def forward(self, x: torch.Tensor, edge_index: torch.Tensor):
        for layer in self.layers:
            if isinstance(layer, torch_geometric.nn.TransformerConv):
                x = layer(x, edge_index)
            else:
                x = layer(x)
        return x


# 4 Transductive learning setting

In [48]:
class TransductiveTrainer:
    def __init__(
        self,
        model: torch.nn.Module,
        optimizer: torch.optim.Optimizer,
        loss_function: torch.nn.Module,
        data: torch_geometric.data.Data,
        epochs: int
    ):
        self.model = model
        self.optimizer = optimizer
        self.loss_function = loss_function
        self.data = data
        self.epochs = epochs
        self.logger = {
            "train_loss": [],
            "train_accuracy": [],
            "val_loss": [],
            "val_accuracy": []
        }

    def fit(self):
        for epoch in range(self.epochs):
            # training step
            self.model.train()
            logits = self.model(self.data.x, self.data.edge_index)
            train_loss_value = self.loss_function(logits[self.data.train_mask], self.data.y[self.data.train_mask])
            self.optimizer.zero_grad()
            train_loss_value.backward()
            self.optimizer.step()
            train_accuracy = self.accuracy(logits[self.data.train_mask].argmax(dim=1), self.data.y[self.data.train_mask])

            train_loss_value = train_loss_value.detach().cpu().numpy()
            train_accuracy = train_accuracy.detach().cpu().numpy()
            print(f"[INFO] epoch {epoch + 1}:\n\ttraining loss: {train_loss_value}")
            print(f"\n\ttraining accuracy: {train_accuracy}")
            self.logger["train_loss"].append(train_loss_value)
            self.logger["train_accuracy"].append(train_accuracy)

            # validation step
            self.model.eval()
            with torch.no_grad():
                logits = self.model(self.data.x, self.data.edge_index)
                val_loss_value = self.loss_function(logits[self.data.val_mask], self.data.y[self.data.val_mask])
                val_accuracy = self.accuracy(logits[self.data.val_mask].argmax(dim=1), self.data.y[self.data.val_mask])

                val_loss_value = val_loss_value.cpu().numpy()
                val_accuracy = val_accuracy.cpu().numpy()
                print(f"\n\tval loss: {val_loss_value}")
                print(f"\n\tval accuracy: {val_accuracy}")
                print(f"\n")
                self.logger["val_loss"].append(val_loss_value)
                self.logger["val_accuracy"].append(val_accuracy)
    
    def accuracy(self, y_predict: torch.Tensor, y_truth: torch.Tensor):
        return torch.sum(y_predict == y_truth) / len(y_truth)

    @torch.no_grad()
    def get_energy(self):
        self.model.eval()
        # Get the current output from the model
        logits = self.model(self.data.x, self.data.edge_index)
        
        # Calculate energy
        energy = self.compute_dirichlet_energy(logits, self.data.edge_index)
        return float(energy.cpu().numpy())

    def compute_dirichlet_energy(self, x: torch.Tensor, edge_index: torch.Tensor):
        row, col = edge_index
        # Normalize node features to unit sphere to ensure energy 
        # is about 'angle' between nodes, not just raw magnitude.
        x_norm = torch.nn.functional.normalize(x, p=2, dim=-1)
        
        # Calculate squared Euclidean distance between all connected node pairs
        source, target = x_norm[row], x_norm[col]
        squared_diff = torch.pow(source - target, 2).sum(dim=-1)
        
        # Average energy per edge
        return squared_diff.mean()
    
    @torch.no_grad()
    def test(self, y_predict: torch.Tensor | None = None, y_truth: torch.Tensor | None = None):
        self.model.eval()
        if y_predict is None and y_truth is None:
            logits = self.model(self.data.x, self.data.edge_index)
            y_predict = logits[self.data.test_mask].argmax(dim=1)
            y_truth = self.data.y[self.data.test_mask]
        test_accuracy = self.accuracy(y_predict, y_truth)

        return float(test_accuracy.cpu().numpy())

# 5 Experiment class

In [53]:
class Experiment:
    def __init__(
        self,
        layers: list[int],
        datasets: list[torch_geometric.data.Dataset],
        epochs: int,
        base_layers: int,
        hyperparameter_grid: dict[str, list[Any]],
        hyperparams_optim_epochs: int
    ):
        self._hyperparameter_grid = hyperparameter_grid
        self._layers = layers
        self._datasets = datasets
        self._base_layers = base_layers
        self._epochs = epochs
        self._hyperparams_optim_epochs = hyperparams_optim_epochs

        self._best_hyperparams_gat = []
        self._best_hyperparams_transformer = []
        self._best_base_accs_gat = []
        self._best_base_accs_trans = []

        self._log = {}
    
    def perform_experiment(self):
        #1. find best hyperparameters on each dataset for both architectures using smaller models
        # (heads, hidden_dim_per_head) are paired in such a way that hidden dimensionality is always the same
        self.optimize_hyperparameters()

        #2. for each dataset, load best hyperparameters for both architectures
        # iterate over provided layers and train both architectures for each layer count
        for dataset_idx, dataset in enumerate(self._datasets):
            metrics = {
                "gat": {
                    "acc": [],
                    "dirichlet": []
                },
                "trans": {
                    "acc": [],
                    "dirichlet": []
                }
            }
            for layer_num in self._layers:
                heads_gat, hidden_dim_gat = self._best_hyperparams_gat[dataset_idx]["heads_hidden_dim"]
                gat_trainer = self.get_trainers(
                    number_of_layers = layer_num,
                    input_size = dataset.num_features,
                    hidden_dim = hidden_dim_gat,
                    output_size = dataset.num_classes,
                    heads = heads_gat,
                    dropout = self._best_hyperparams_gat[dataset_idx]["dropout"],
                    lr = self._best_hyperparams_gat[dataset_idx]["lr"],
                    weight_decay = self._best_hyperparams_gat[dataset_idx]["weight_decay"],
                    data = dataset[0],
                    epochs = self._epochs,
                    model_type = "gat"
                )[0]

                heads_trans, hidden_dim_trans = self._best_hyperparams_transformer[dataset_idx]["heads_hidden_dim"]
                trans_trainer = self.get_trainers(
                    number_of_layers = layer_num,
                    input_size = dataset.num_features,
                    hidden_dim = hidden_dim_trans,
                    output_size = dataset.num_classes,
                    heads = heads_trans,
                    dropout = self._best_hyperparams_transformer[dataset_idx]["dropout"],
                    lr = self._best_hyperparams_transformer[dataset_idx]["lr"],
                    weight_decay = self._best_hyperparams_transformer[dataset_idx]["weight_decay"],
                    data = dataset[0],
                    epochs = self._epochs,
                    model_type = "trans"
                )[0]

                gat_trainer.fit()
                trans_trainer.fit()
                gat_accuracy = gat_trainer.test()
                trans_accuracy = trans_trainer.test()
                gat_dirichlet = gat_trainer.get_energy()
                trans_dirichlet = trans_trainer.get_energy()

                metrics["gat"]["acc"].append(gat_accuracy)
                metrics["trans"]["acc"].append(trans_accuracy)
                metrics["gat"]["dirichlet"].append(gat_dirichlet)
                metrics["trans"]["dirichlet"].append(trans_dirichlet)
            
            self._log[dataset.name] = metrics

            return self._log

        #3. perform suitable comparison on each dataset, for each layer count

    def optimize_hyperparameters(self):
        for dataset in self._datasets:
            best_gat, best_trans, best_gat_accuracy, best_trans_accuracy = self.grid_search(dataset)
            self._best_hyperparams_gat.append(best_gat)
            self._best_hyperparams_transformer.append(best_trans)
            self._best_base_accs_gat.append(best_gat_accuracy)
            self._best_base_accs_trans.append(best_trans_accuracy)
    
    def get_combination_generator(self):
        keys = self._hyperparameter_grid.keys()
        values = self._hyperparameter_grid.values()
        hyperparams_combinations = itertools.product(*values)
        for combination in hyperparams_combinations:
            yield dict(zip(keys, combination))
    
    def get_trainers(
        self,
        number_of_layers: int,
        input_size: int,
        hidden_dim: int,
        output_size: int,
        heads: int,
        dropout: float,
        lr: float,
        weight_decay: float,
        data: torch_geometric.data.Data,
        epochs: int,
        model_type: str | None = None
    ):
        trainers = []

        if model_type == "gat" or model_type == None:
            gat_base = GATNetwork(
                number_of_layers = number_of_layers,
                input_size = input_size,
                hidden_size = hidden_dim,
                output_size = output_size,
                num_heads = heads,
                dropout_rate = dropout
            ).to(DEVICE)
            optimizer_gat = torch.optim.Adam(
                params = gat_base.parameters(),
                lr = lr,
                weight_decay = weight_decay
            )
            gat_trainer = TransductiveTrainer(
                model = gat_base,
                optimizer = optimizer_gat,
                loss_function = torch.nn.CrossEntropyLoss(),
                data = data,
                epochs = epochs
            )
            trainers.append(gat_trainer)

        if model_type == "trans" or model_type == None:
            trans_base = TransformerConvNetwork(
                number_of_layers = number_of_layers,
                input_size = input_size,
                hidden_size = hidden_dim,
                output_size = output_size,
                num_heads = heads,
                dropout_rate = dropout
            ).to(DEVICE)
            optimizer_trans = torch.optim.Adam(
                params = trans_base.parameters(),
                lr = lr,
                weight_decay = weight_decay
            )
            trans_trainer = TransductiveTrainer(
                model = trans_base,
                optimizer = optimizer_trans,
                loss_function = torch.nn.CrossEntropyLoss(),
                data = data,
                epochs = epochs
            )
            trainers.append(trans_trainer)

        return trainers   

    def grid_search(
        self,
        dataset: torch_geometric.data.Dataset
    ):
        hyperparameter_generator = self.get_combination_generator()

        best_gat_accuracy = 0.0
        best_trans_accuracy = 0.0
        best_gat = None
        best_trans = None

        for hyperparameters in hyperparameter_generator:
            heads, hidden_dim = hyperparameters["heads_hidden_dim"]
            
            gat_trainer, trans_trainer = self.get_trainers(
                number_of_layers = self._base_layers,
                input_size = dataset.num_features,
                hidden_dim = hidden_dim,
                output_size = dataset.num_classes,
                heads = heads,
                dropout = hyperparameters["dropout"],
                lr = hyperparameters["lr"],
                weight_decay = hyperparameters["weight_decay"],
                data = dataset[0],
                epochs = self._hyperparams_optim_epochs
            )

            gat_trainer.fit()
            trans_trainer.fit()
            gat_accuracy = gat_trainer.test()
            trans_accuracy = trans_trainer.test()
            if gat_accuracy > best_gat_accuracy:
                best_gat_accuracy = gat_accuracy
                best_gat = hyperparameters
            if trans_accuracy > best_trans_accuracy:
                best_trans_accuracy = trans_accuracy
                best_trans = hyperparameters
        
        return best_gat, best_trans, best_gat_accuracy, best_trans_accuracy

In [54]:
experiment = Experiment(
    layers = [1, 2, 3],
    datasets = [pubmed_dataset.to(DEVICE)],
    epochs = 2,
    base_layers = 2,
    hyperparameter_grid = {
        "heads_hidden_dim": [(4, 32), (2, 64)],
        "dropout": [0.1, 0.2, 0.3],
        "lr": [1e-2, 5e-2, 1e-3],
        "weight_decay": [5e-6, 5e-5]
    },
    hyperparams_optim_epochs = 2
)

In [55]:
log = experiment.perform_experiment()

[INFO] epoch 1:
	training loss: 1.3084501028060913

	training accuracy: 0.21666668355464935

	val loss: 0.7448215484619141

	val accuracy: 0.6940000057220459


[INFO] epoch 2:
	training loss: 0.2722863554954529

	training accuracy: 0.9000000357627869

	val loss: 0.6332069039344788

	val accuracy: 0.718000054359436


[INFO] epoch 1:
	training loss: 1.107550859451294

	training accuracy: 0.31666669249534607

	val loss: 0.704153835773468

	val accuracy: 0.7140000462532043


[INFO] epoch 2:
	training loss: 0.17669863998889923

	training accuracy: 0.9666666984558105

	val loss: 0.6809325218200684

	val accuracy: 0.7360000610351562


[INFO] epoch 1:
	training loss: 1.1321007013320923

	training accuracy: 0.36666667461395264

	val loss: 0.7261190414428711

	val accuracy: 0.6860000491142273


[INFO] epoch 2:
	training loss: 0.20172351598739624

	training accuracy: 0.9500000476837158

	val loss: 0.6956754326820374

	val accuracy: 0.7120000123977661


[INFO] epoch 1:
	training loss: 1.1917986869

In [56]:
print(log)

{'PubMed': {'gat': {'acc': [0.6760000586509705, 0.7450000643730164, 0.7260000109672546], 'dirichlet': [0.40797993540763855, 0.19336554408073425, 0.12499809265136719]}, 'trans': {'acc': [0.6790000200271606, 0.6910000443458557, 0.7070000171661377], 'dirichlet': [0.5497143268585205, 0.2900509834289551, 0.2880724370479584]}}}
